this code relates to extraction and analysis of audio feature, done in the scope of an experiment on musical improvisation, mental states and time perception.

---------- EXPERIMENT DESCRIPTION ------------

1. Research Question - does musical improvisation affect musicians time perception? 2 dimensions of improvisation: Group vs solo; familiar harmony (A) vs unfamiliar harmony (B) vs Free (C). 6 tasks per participant. groups included 4 musicians
2. Data Collection - answers about time perception enjoyment, difficulty etc + recordings of improvisations
3. Feature Extraction - at a point in the analysis data suggested musical features could be driving changes in perceived duration - musical tempo, and how clear it could be perceived could be possible factors. To confirm this we extracted BPM and beat confidence using Essentia
4. Statistical Analysis - first simple T-tests and one-way anovas compared bpm and beat confidence across the involved musical conditions. Later, in full paper, we used Linear-Mixed models to see if these audio features predicted Time perception.
5. Results - there were indeed differences in beat confidence, not BPM, but in fact these did not predict duration estimates



In [41]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
from sklearn.linear_model import LinearRegression

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from scipy.stats import shapiro
from scipy.stats import mannwhitneyu

#for mixed models and anovas
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pingouin as pg

#for MIR
import os
import essentia.standard as es

In [42]:
# loading dataset with participants's answers for each experimental trial

"""df = pd.read_csv('improv-time 2 - clean_columns_added_relative_estimates.csv', sep = ",")
df.head()"""

'df = pd.read_csv(\'improv-time 2 - clean_columns_added_relative_estimates.csv\', sep = ",")\ndf.head()'

In [43]:
# add path for each audio file into csv. Audio files were labelled according to the experimental condition, which can be extracted by the variables in each row/trial

"""df["recording_path"] = ""

for index, row in df.iterrows():
    if (row["Group-Solo"] == "Group"):
        df.loc[index, "recording_path"] = "recordings/Groups/" + row["Assigned_group"] + " " + row["Type-improv"] + " " + row["Formato audio"]
        

for index, row in df.iterrows():
    if (row["Group-Solo"] == "Solo"):
        df.loc[index, "recording_path"] = "recordings/Solos/" + row["ID de participante"] + " Solo " + row["Type-improv"] + " " + row["Formato audio"]

df.to_csv("improv-time 3 - added recording paths.csv")"""

'df["recording_path"] = ""\n\nfor index, row in df.iterrows():\n    if (row["Group-Solo"] == "Group"):\n        df.loc[index, "recording_path"] = "recordings/Groups/" + row["Assigned_group"] + " " + row["Type-improv"] + " " + row["Formato audio"]\n\n\nfor index, row in df.iterrows():\n    if (row["Group-Solo"] == "Solo"):\n        df.loc[index, "recording_path"] = "recordings/Solos/" + row["ID de participante"] + " Solo " + row["Type-improv"] + " " + row["Formato audio"]\n\ndf.to_csv("improv-time 3 - added recording paths.csv")'

In [44]:
#    estimate BPM and beat confidence, and store values in the csv, in new columns, and also as arrays for statistical comparison in this same file 

"""
import os
import subprocess
import tempfile
import numpy as np
import essentia.standard as es

df["BPM_estimated"] = np.nan
df["beat_confidence"] = np.nan
df["onset_rate"] = np.nan
df["dynamic_variability"] = np.nan
df["spectral_complexity"] = np.nan


# ---------------------------------------------------------
# FUNCTION: convert any audio file to standardized WAV
# mono / 44.1kHz / PCM 16-bit
# ---------------------------------------------------------

def standardize_audio(input_path):

    temp_wav = tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False
    )

    output_path = temp_wav.name
    temp_wav.close()

    command = [
        "ffmpeg",
        "-y",
        "-i", input_path,
        "-ac", "1",
        "-ar", "44100",
        "-sample_fmt", "s16",
        output_path
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    print("\n-----------------------------------")
    print("FFMPEG INPUT :", input_path)
    print("FFMPEG OUTPUT:", output_path)
    print("RETURN CODE  :", result.returncode)

    if result.returncode != 0:
        print("\nFFMPEG STDERR:")
        print(result.stderr)

    if os.path.exists(output_path):
        print("OUTPUT EXISTS:", True)
        print("OUTPUT SIZE  :", os.path.getsize(output_path), "bytes")
    else:
        print("OUTPUT EXISTS:", False)

    print("-----------------------------------\n")

    return output_path



for index, row in df.iterrows():

    full_path = df.loc[index, "recording_path"]

    print("\n===================================")
    print("ROW INDEX:", index)
    print("SOURCE FILE:", full_path)
    print("FILE EXISTS:", os.path.exists(full_path))

    if os.path.exists(full_path):
        print("FILE SIZE:", os.path.getsize(full_path), "bytes")

    print("===================================\n")


    try:

        # ---------------------------------------------
        # STANDARDIZE AUDIO
        # ---------------------------------------------
        standardized_file = standardize_audio(full_path)

        print("TEMP FILE:", standardized_file)
        print("TEMP EXISTS:", os.path.exists(standardized_file))

        # ---------------------------------------------
        # LOAD STANDARDIZED AUDIO
        # ---------------------------------------------
        audio = es.MonoLoader(
            filename=standardized_file,
            sampleRate=44100
        )()

        print("shape:", audio.shape)
        print("min/max:", np.min(audio), np.max(audio))
        print("nan:", np.isnan(audio).any())

        # -------------------------------------------------------------
        # ADDITION: Onset density (events / second)
        # -------------------------------------------------------------

        onset_detector = es.OnsetRate()

        onset_times, onset_rate = onset_detector(audio)

        df.loc[index, "onset_rate"] = float(onset_rate)

        # -------------------------------------------------------------
        # ADDITION: Dynamic variability
        # -------------------------------------------------------------

        frame_generator = es.FrameGenerator(
            audio,
            frameSize=2048,
            hopSize=1024
        )

        rms_algorithm = es.RMS()

        rms_values = []

        for frame in frame_generator:
            rms_values.append(rms_algorithm(frame))

        df.loc[index, "dynamic_variability"] = np.std(rms_values)  

        # -------------------------------------------------------------
        # ADDITION: Spectral complexity
        # -------------------------------------------------------------

        window = es.Windowing(type='hann')
        spectrum = es.Spectrum()
        spectral_complexity = es.SpectralComplexity()

        complexities = []

        for frame in es.FrameGenerator(audio,
                                    frameSize=2048,
                                    hopSize=1024):

            spec = spectrum(window(frame))
            complexities.append(
                spectral_complexity(spec)
            )

        df.loc[index, "spectral_complexity"] = np.mean(complexities)                          

        # ---------------------------------------------
        # RHYTHM EXTRACTION
        # ---------------------------------------------
        rhythm_extractor = es.RhythmExtractor2013(
            method="multifeature"
        )

        bpm, beats, beat_confidence, _, _ = rhythm_extractor(audio)

        df.loc[index, "BPM_estimated"] = bpm
        df.loc[index, "beat_confidence"] = beat_confidence

        print(df.loc[index,
             ["BPM_estimated",
              "beat_confidence",
              "onset_rate",
              "dynamic_variability",
              "spectral_complexity"]])

        # ---------------------------------------------
        # DELETE TEMP FILE
        # ---------------------------------------------
        os.remove(standardized_file)

    except Exception as e:

        print("ERROR:", e)

        if 'standardized_file' in locals():
            print(
                "TEMP FILE EXISTS:",
                os.path.exists(standardized_file)
            )

            if os.path.exists(standardized_file):
                print(
                    "TEMP FILE SIZE:",
                    os.path.getsize(standardized_file),
                    "bytes"
                )
df.to_csv("improv-time results 4 - final.csv")"""


'\nimport os\nimport subprocess\nimport tempfile\nimport numpy as np\nimport essentia.standard as es\n\ndf["BPM_estimated"] = np.nan\ndf["beat_confidence"] = np.nan\ndf["onset_rate"] = np.nan\ndf["dynamic_variability"] = np.nan\ndf["spectral_complexity"] = np.nan\n\n\n# ---------------------------------------------------------\n# FUNCTION: convert any audio file to standardized WAV\n# mono / 44.1kHz / PCM 16-bit\n# ---------------------------------------------------------\n\ndef standardize_audio(input_path):\n\n    temp_wav = tempfile.NamedTemporaryFile(\n        suffix=".wav",\n        delete=False\n    )\n\n    output_path = temp_wav.name\n    temp_wav.close()\n\n    command = [\n        "ffmpeg",\n        "-y",\n        "-i", input_path,\n        "-ac", "1",\n        "-ar", "44100",\n        "-sample_fmt", "s16",\n        output_path\n    ]\n\n    result = subprocess.run(\n        command,\n        capture_output=True,\n        text=True\n    )\n\n    print("\n---------------------

Import csv that now includes these estimates and perform LMM and Wilcoxon rank (if residuls are non-normal) to confirm whether these musical features are responsible for changes found in DE between group/solo

In [45]:
df2 = pd.read_csv("improv-time results 4 - final.csv")
df2.dtypes

Unnamed: 0.1                                                                                          int64
Unnamed: 0                                                                                            int64
data_collection                                                                                      object
ID de participante                                                                                   object
Tipo de instrumento                                                                                  object
G01Q06. Escolha o ID da sessão                                                                       object
Assigned_group                                                                                       object
Group-Solo                                                                                           object
Type-improv                                                                                          object
Real_duration(s)            

In [46]:
df2["SQRT_relative_estimate"] = np.sqrt(df2["relative_estimate"])
df2 = df2.dropna()

In [49]:
def LMM (covariate):

    formula = (
        f'Q("SQRT_relative_estimate") ~ '
        f'C(Q("Group-Solo")) + Q("{covariate}")'
    )

    md_eng = smf.mixedlm(
    formula,
    data=df2,
    groups=df2["ID de participante"]
    )
    results_eng = md_eng.fit()
    print(results_eng.summary())

In [50]:
musical_features = ["BPM_estimated","beat_confidence","onset_rate", "dynamic_variability", "spectral_complexity"]

for i in musical_features:
    LMM(i)

                  Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: Q("SQRT_relative_estimate")
No. Observations: 270     Method:             REML                       
No. Groups:       48      Scale:              0.0193                     
Min. group size:  3       Log-Likelihood:     87.5380                    
Max. group size:  6       Converged:          Yes                        
Mean group size:  5.6                                                    
-------------------------------------------------------------------------
                               Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------------------
Intercept                       1.198    0.063 18.916 0.000  1.074  1.322
C(Q("Group-Solo"))[T.Solo]     -0.045    0.018 -2.531 0.011 -0.080 -0.010
Q("BPM_estimated")              0.000    0.000  0.175 0.861 -0.001  0.001
Group Var                       0.026    0.048          